### Frontier Engineer finder
This notebook takes the Trainings report from Partner Center Insights, and outputs information relating to the Frontier Engineer Badge.

In [9]:
import pandas as pd

filename = 'AvanadeTrainings.csv'  # Update with your actual filename

if filename.lower().endswith('.tsv'):
    df = pd.read_csv(filename, sep='\t')
elif filename.lower().endswith('.csv'):
    df = pd.read_csv(filename)
else:
    raise ValueError("Unsupported file type. Use a .csv or .tsv file.")

print(f"Loaded {len(df):,} training activities")
unique_learners = df['AADUserId'].nunique()
print(f"Loaded {unique_learners:,} unique learners")


Loaded 153,449 training activities
Loaded 15,024 unique learners


### Learners with all three certifications

In [10]:
target_certs = [
    'Microsoft Certified: Azure AI Engineer Associate',
    'GitHub Copilot',
    'Microsoft Certified: Agentic AI Business Solutions Architect'
]

# Filter for target certifications that are Active and of type Certification, with valid Email
filtered = df[
    (df['TrainingTitle'].isin(target_certs)) &
    (df['ActivationStatus'] == 'Active') &
    (df['TrainingType'] == 'Certification') &
    (df['Email'].notna()) &
    (df['Email'].str.strip() != '')
]

# Find learners who have earned all 3 certifications
cert_counts = filtered.groupby('Email')['TrainingTitle'].nunique()
qualified_ids = cert_counts[cert_counts == 3].index

# Build output table with one row per learner
qualified = filtered[filtered['Email'].isin(qualified_ids)]
result = (
    qualified.groupby('Email')
    .agg(
        FirstName=('IndividualFirstName', 'first'),
        LastName=('IndividualLastName', 'first'),
        CorpEmail=('CorpEmail', 'first')
    )
    .reset_index()
)
result['Name'] = result['FirstName'].str.cat(result['LastName'], sep=' ').str.strip()
result = result[['Name', 'CorpEmail']]

print(f'Learners with all 3 certifications (Active): {len(result)}')
result

Learners with all 3 certifications (Active): 7


,Name,CorpEmail
0,Eddie Liao,eddie.liao@avanade.com
1,Bochao Zhang,bochao.zhang@avanade.com
2,Digvijay Chauhan,digvijay.chauhan@avanade.com
3,Kritdikoon Woraitthinan,k.woraitthinan@avanade.com
4,Carolina Alves de Oliveira e Silva,c.oliveira.e.silva@avanade.com
5,Renan Evangelista Pereira,renan.pereira@avanade.com
6,Ryan Randhawa,ryan.s.randhawa@avanade.com


### Learners with exactly 2 out of 3 certifications

In [11]:
two_of_three_ids = cert_counts[cert_counts == 2].index
two_of_three = filtered[filtered['Email'].isin(two_of_three_ids)]

# For each learner, find the missing certification
rows = []
for email, group in two_of_three.groupby('Email'):
    earned = set(group['TrainingTitle'])
    missing = set(target_certs) - earned
    row = group.iloc[0]
    name = f"{row['IndividualFirstName']} {row['IndividualLastName']}".strip()
    for cert in missing:
        rows.append({'Name': name, 'CorpEmail': row['CorpEmail'], 'Missing Certification': cert})

result_2of3 = pd.DataFrame(rows)

print(f'Learners with 2 out of 3 certifications (Active): {result_2of3["CorpEmail"].nunique()}')
result_2of3

Learners with 2 out of 3 certifications (Active): 78


,Name,CorpEmail,Missing Certification
0,Andrew Hiron,a.hiron@avanade.com,Microsoft Certified: Agentic AI Business Solut...
1,Ajay Kumar Y,ajay.kumar.y@accenture.com,Microsoft Certified: Agentic AI Business Solut...
2,Amanda Poupko,amanda.poupko@avanade.com,Microsoft Certified: Agentic AI Business Solut...
3,Anoop Sukumaran,anoop.p.sukumaran@accenture.com,GitHub Copilot
4,Tony Skidmore,NaN,Microsoft Certified: Agentic AI Business Solut...
...,...,...,...
79,viaminds@hotmail.com nan,cesar.calvo.cobo@avanade.com,GitHub Copilot
80,Vikingur Saemundsson,vikingur.saemundsson@avanade.com,GitHub Copilot
81,Vishali Ramalingam,vishali.ramalingam@accenture.com,Microsoft Certified: Agentic AI Business Solut...
82,Yoichiro Abe,yoichiro.abe@avanade.com,GitHub Copilot


### At risk certifications
All target certifications, in order of expiration date, earliest to latest

In [12]:
# At risk certifications
at_risk = filtered.copy()
at_risk['ExpirationDate'] = pd.to_datetime(at_risk['ExpirationDate'])
at_risk = at_risk.dropna(subset=['ExpirationDate'])
at_risk = at_risk.sort_values('ExpirationDate')

at_risk['Name'] = at_risk['IndividualFirstName'].str.cat(at_risk['IndividualLastName'], sep=' ').str.strip()
at_risk = at_risk[['Name', 'CorpEmail', 'TrainingTitle', 'ExpirationDate']].rename(
    columns={'TrainingTitle': 'Certification', 'ExpirationDate': 'Expiration Date'}
)
at_risk = at_risk.reset_index(drop=True)

today = pd.Timestamp.today().normalize()
cutoff = today + pd.DateOffset(months=6)

at_risk = at_risk[
    (at_risk['Expiration Date'] >= today) &
    (at_risk['Expiration Date'] <= cutoff)
].reset_index(drop=True)

print(f'Certifications expiring within 6 months: {len(at_risk)}')
at_risk

Certifications expiring within 6 months: 215


,Name,CorpEmail,Certification,Expiration Date
0,Shefali Kukreja,shefali.b.kukreja@avanade.com,Microsoft Certified: Azure AI Engineer Associate,2026-05-13
1,Ludovica Binetti,ludovica.binetti@avanade.com,Microsoft Certified: Azure AI Engineer Associate,2026-05-13
2,NSR Krishna Sombhotla,NaN,Microsoft Certified: Azure AI Engineer Associate,2026-05-14
3,Sathya Devi,sathya.devi@avanade.com,Microsoft Certified: Azure AI Engineer Associate,2026-05-16
4,Mohan Raj Ms,mohan.raj.ms@accenture.com,Microsoft Certified: Azure AI Engineer Associate,2026-05-17
...,...,...,...,...
210,Amanda Poupko,amanda.poupko@avanade.com,Microsoft Certified: Azure AI Engineer Associate,2026-11-08
211,Ashutosh Kumar,ashutosh.dt.kumar@accenture.com,Microsoft Certified: Azure AI Engineer Associate,2026-11-08
212,Mahmoud Sabry Eldesouky Abdelhady Aly,mahmoud.sabry.e.aly@avanade.com,Microsoft Certified: Azure AI Engineer Associate,2026-11-10
213,Shakiya Friend,shakiya.friend@avanade.com,Microsoft Certified: Azure AI Engineer Associate,2026-11-11
